In [ ]:
%load_ext autoreload
%autoreload 2

In [1]:
import os
import tqdm
import numpy as np

from linalgo.hub import BQClient

from wsd.models import JMDict


In [2]:
task_id = os.getenv('LINHUB_TASK')
client = BQClient(task_id)
jmdict = JMDict()

DefaultCredentialsError: Your default credentials were not found. To set up Application Default Credentials, see https://cloud.google.com/docs/authentication/external/set-up-adc for more information.

In [ ]:
annotations = client.get_annotations()
docs = client.get_documents()
docs = [doc for doc in docs if len(doc.annotations) > 0]

In [ ]:
def get_offset(token):
    start = token.target.selector[0].start_offset
    end = token.target.selector[0].end_offset
    return start, end

y_pred = []
y_true = []
for doc in tqdm.tqdm(docs):
    yp = jmdict.predict(doc.content)
    yp = [y for y in yp if y is not None]
    yt = [a.body for a in sorted(doc.annotations, key=get_offset)]
    # TODO: Will break when more than one annotator
    if len(yt) == len(yp):  # Some documents are partially annotated.
        y_pred.extend(yp)
        y_true.extend(yt)
y_true = np.array(y_true)
y_pred = np.array(y_pred)


In [ ]:
acc = np.sum(y_true == y_pred) / len(y_true)
print(f"accuracy = {acc:%}")